[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch04/NN_DL_ch04_MNIST/NN_DL_ch04_MNIST.ipynb)

# NN_DL_ch04_MNIST

Train a CNN on MNIST.

**Author:** Daniel Traian Pele


In [ ]:
!pip install torch torchvision matplotlib scikit-learn -q


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,classification_report
torch.manual_seed(42);np.random.seed(42)
BLUE,RED,GREEN,CRIM="#1A3A6E","#CD0000","#2E7D32","#DC3545"
dev=torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load MNIST


In [ ]:
tf=T.Compose([T.ToTensor(),T.Normalize((0.1307,),(0.3081,))])
tr=torchvision.datasets.MNIST("./data",train=True,download=True,transform=tf)
te=torchvision.datasets.MNIST("./data",train=False,download=True,transform=tf)
trl=DataLoader(tr,64,shuffle=True);tel=DataLoader(te,1000)


## CNN


In [ ]:
class CNN(nn.Module):
  def __init__(s):
    super().__init__()
    s.c1=nn.Conv2d(1,32,3,padding=1);s.c2=nn.Conv2d(32,64,3,padding=1)
    s.p=nn.MaxPool2d(2);s.f1=nn.Linear(64*7*7,128);s.f2=nn.Linear(128,10);s.d=nn.Dropout(.25)
  def forward(s,x):
    x=s.p(F.relu(s.c1(x)));x=s.p(F.relu(s.c2(x)))
    return s.f2(s.d(F.relu(s.f1(x.view(-1,64*7*7)))))
m=CNN().to(dev);print(sum(p.numel() for p in m.parameters()),"params")


## Train


In [ ]:
o=torch.optim.Adam(m.parameters(),1e-3);c=nn.CrossEntropyLoss()
L,A=[],[]
for e in range(10):
  m.train();rl=0
  for x,y in trl:x,y=x.to(dev),y.to(dev);o.zero_grad();l=c(m(x),y);l.backward();o.step();rl+=l.item()
  L.append(rl/len(trl));m.eval();cor=tot=0
  with torch.no_grad():
    for x,y in tel:x,y=x.to(dev),y.to(dev);_,p=m(x).max(1);tot+=y.size(0);cor+=p.eq(y).sum().item()
  A.append(100*cor/tot);print(f"E{e+1} L={L[-1]:.4f} A={A[-1]:.2f}%")


## Results


In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4.5))
a1.plot(range(1,11),L,color=BLUE,lw=2,marker="o",ms=4);a1.set_xlabel("Epoch");a1.set_ylabel("Loss");a1.set_title("Training Loss",fontweight="bold");a1.set_facecolor("none")
a2.plot(range(1,11),A,color=RED,lw=2,marker="s",ms=4);a2.set_xlabel("Epoch");a2.set_ylabel("Acc %");a2.set_title("Test Accuracy",fontweight="bold");a2.set_facecolor("none")
fig.patch.set_alpha(0);plt.tight_layout()
plt.savefig("nn_ch04_mnist_training.pdf",bbox_inches="tight",dpi=300,transparent=True)
plt.savefig("nn_ch04_mnist_training.png",bbox_inches="tight",dpi=150,transparent=True);plt.show()


In [ ]:
m.eval();ap,al=[],[]
with torch.no_grad():
  for x,y in tel:_,p=m(x.to(dev)).max(1);ap.extend(p.cpu().numpy());al.extend(y.numpy())
cm=confusion_matrix(al,ap)
fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(cm,cmap="Blues")
ax.set_xticks(range(10));ax.set_yticks(range(10));ax.set_xlabel("Pred");ax.set_ylabel("True")
ax.set_title("MNIST Confusion Matrix",fontsize=13,fontweight="bold",color=BLUE)
for i in range(10):
  for j in range(10):ax.text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=9)
plt.colorbar(im,ax=ax,shrink=.8);fig.patch.set_alpha(0);plt.tight_layout()
plt.savefig("nn_ch04_mnist_confusion.pdf",bbox_inches="tight",dpi=300,transparent=True)
plt.savefig("nn_ch04_mnist_confusion.png",bbox_inches="tight",dpi=150,transparent=True);plt.show()
print(classification_report(al,ap))
